> Public repository edition. The analysis uses the included **sample** by default. Full report results used the separate full dataset. Code paths and validation were cleaned for publication; original score formulas, pair split, model settings and evaluation definitions are retained. Outputs are cleared. See [reproducibility notes](../docs/reproducibility.md).

# Calibration

## Model 1: Logistic calibration using `r_overlap`

Training the first probability model using r_overlap as the only input feature

In [ ]:
from pathlib import Path
import sys

# Start Jupyter in the repository root or its notebooks/ directory.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "project_config.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_config import get_paths

paths = get_paths()
print("Data mode:", paths.mode)
print("Input:", paths.input_path)
print("Local processed data:", paths.processed_dir)
print("Local evaluation outputs:", paths.evaluation_dir)

# Loading packages and train/test datasets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
    roc_auc_score,
    average_precision_score
)
from sklearn.calibration import calibration_curve

In [ ]:
train_path = paths.processed_dir / "IdentityMF_MinMax_train.csv"
test_path = paths.processed_dir / "IdentityMF_MinMax_test.csv"
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError("Run 01_preprocess.ipynb in the same data mode first.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
if set(train_df["pair_id"]) & set(test_df["pair_id"]):
    raise ValueError("A pair_id appears in both train and test.")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Defining the input feature and target variable.

In [ ]:
feature = "r_overlap"
target = "label"

X_train = train_df[[feature]]
y_train = train_df[target]

X_test = test_df[[feature]]
y_test = test_df[target]

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True).sort_index())

We now train the logistic calibration model. Since the goal is probability calibration, we do not use class weighting here. The model should learn probabilities based on the real class proportions in the training data.

The model estimates:

$$
P(y=1 \mid r)
$$

where $r$ is r_overlap and $y=1$ means same-compound comparison.

In [ ]:
logistic_r_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)

logistic_r_model.fit(X_train, y_train)

intercept = logistic_r_model.intercept_[0]
coefficient = logistic_r_model.coef_[0][0]

print("Intercept:", intercept)
print("Coefficient for r_overlap:", coefficient)

Predicting same-compound probabilities on the test set

In [ ]:
test_pred_prob = logistic_r_model.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(test_pred_prob[:10])

print("\nPredicted probability summary:")
print(pd.Series(test_pred_prob).describe())

Converting probabilities to class labels using a 0.5 threshold

In [ ]:
threshold = 0.5

test_pred_class = (test_pred_prob >= threshold).astype(int)

print("Predicted class counts:")
print(pd.Series(test_pred_class).value_counts().sort_index())

Evaluating classification and probability performance

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, test_pred_class).ravel()

specificity = tn / (tn + fp)
false_positive_rate = fp / (fp + tn)

metrics = {
    "accuracy": accuracy_score(y_test, test_pred_class),
    "precision": precision_score(y_test, test_pred_class, zero_division=0),
    "recall": recall_score(y_test, test_pred_class),
    "specificity": specificity,
    "false_positive_rate": false_positive_rate,
    "f1_score": f1_score(y_test, test_pred_class),
    "brier_score": brier_score_loss(y_test, test_pred_prob),
    "log_loss": log_loss(y_test, test_pred_prob),
    "roc_auc": roc_auc_score(y_test, test_pred_prob),
    "pr_auc": average_precision_score(y_test, test_pred_prob)
}

metrics_df = pd.DataFrame(metrics, index=["Logistic_r_overlap"]).T

metrics_df

Checking the confusion matrix at threshold 0.5

In [ ]:
conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, test_pred_class),
    index=["Actual different (0)", "Actual same (1)"],
    columns=["Predicted different (0)", "Predicted same (1)"]
)

conf_matrix

Checking whether predicted probabilities match observed frequencies

In [ ]:
prob_true, prob_pred = calibration_curve(
    y_test,
    test_pred_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(6, 6))

plt.plot(prob_pred, prob_true, marker="o", label="Logistic r_overlap")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction of same-compound rows")
plt.title("Calibration curve: logistic model using r_overlap")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Plotting the learned probability curve

In [ ]:
r_grid = np.linspace(
    train_df["r_overlap"].min(),
    train_df["r_overlap"].max(),
    500
).reshape(-1, 1)

prob_grid = logistic_r_model.predict_proba(pd.DataFrame(r_grid, columns=["r_overlap"]))[:, 1]

plt.figure(figsize=(8, 5))

plt.plot(r_grid, prob_grid)

plt.xlabel("r_overlap")
plt.ylabel("Predicted probability of same compound")
plt.title("Learned logistic calibration curve")
plt.grid(alpha=0.3)
plt.show()

Saving test-set predictions for later comparison

In [ ]:
predictions_df = test_df.copy()

predictions_df["logistic_r_probability"] = test_pred_prob
predictions_df["logistic_r_prediction"] = test_pred_class

output_path = paths.processed_dir / "IdentityMF_Logistic_r_overlap_test_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

predictions_df.to_csv(output_path, index=False)

print("Predictions saved to:", output_path)

## Model 2: Isotonic calibration

In this section, we train an isotonic calibration model using r_overlap. Unlike logistic regression, isotonic regression does not assume a specific curve shape. It only assumes that larger r_overlap values should not lead to lower same-compound probabilities.

In [ ]:
from sklearn.isotonic import IsotonicRegression

Defining one-dimensional inputs for isotonic regression

In [ ]:
X_train_iso = train_df["r_overlap"]
y_train_iso = train_df["label"]

X_test_iso = test_df["r_overlap"]
y_test_iso = test_df["label"]

print("Training input shape:", X_train_iso.shape)
print("Testing input shape:", X_test_iso.shape)

Training isotonic regression with out-of-range values clipped

In [ ]:
isotonic_r_model = IsotonicRegression(
    y_min=0,
    y_max=1,
    increasing=True,
    out_of_bounds="clip"
)

isotonic_r_model.fit(X_train_iso, y_train_iso)

Predicting test-set probabilities using isotonic regression

In [ ]:
test_iso_prob = isotonic_r_model.predict(X_test_iso)

print("First 10 isotonic predicted probabilities:")
print(test_iso_prob[:10])

print("\nPredicted probability summary:")
print(pd.Series(test_iso_prob).describe())

Converting isotonic probabilities to class labels using threshold 0.5

In [ ]:
threshold = 0.5

test_iso_class = (test_iso_prob >= threshold).astype(int)

print("Predicted class counts:")
print(pd.Series(test_iso_class).value_counts().sort_index())

Evaluating isotonic model performance on the test set

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test_iso, test_iso_class).ravel()

specificity = tn / (tn + fp)
false_positive_rate = fp / (fp + tn)

isotonic_metrics = {
    "accuracy": accuracy_score(y_test_iso, test_iso_class),
    "precision": precision_score(y_test_iso, test_iso_class, zero_division=0),
    "recall": recall_score(y_test_iso, test_iso_class),
    "specificity": specificity,
    "false_positive_rate": false_positive_rate,
    "f1_score": f1_score(y_test_iso, test_iso_class),
    "brier_score": brier_score_loss(y_test_iso, test_iso_prob),
    "log_loss": log_loss(y_test_iso, np.clip(test_iso_prob, 1e-15, 1 - 1e-15)),
    "roc_auc": roc_auc_score(y_test_iso, test_iso_prob),
    "pr_auc": average_precision_score(y_test_iso, test_iso_prob)
}

isotonic_metrics_df = pd.DataFrame(
    isotonic_metrics,
    index=["Isotonic_r_overlap"]
).T

isotonic_metrics_df

Checking the isotonic confusion matrix at threshold 0.5

In [ ]:
iso_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test_iso, test_iso_class),
    index=["Actual different (0)", "Actual same (1)"],
    columns=["Predicted different (0)", "Predicted same (1)"]
)

iso_conf_matrix

Comparing logistic and isotonic model metrics

In [ ]:
try:
    comparison_metrics_df = pd.concat(
        [metrics_df, isotonic_metrics_df],
        axis=1
    )
except NameError:
    comparison_metrics_df = isotonic_metrics_df.copy()

comparison_metrics_df

Plotting calibration curves for model comparison

In [ ]:
iso_prob_true, iso_prob_pred = calibration_curve(
    y_test_iso,
    test_iso_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(6, 6))

try:
    log_prob_true, log_prob_pred = calibration_curve(
        y_test,
        test_pred_prob,
        n_bins=10,
        strategy="quantile"
    )
    plt.plot(
        log_prob_pred,
        log_prob_true,
        marker="o",
        label="Logistic r_overlap"
    )
except NameError:
    pass

plt.plot(
    iso_prob_pred,
    iso_prob_true,
    marker="o",
    label="Isotonic r_overlap"
)

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction of same-compound rows")
plt.title("Calibration curve comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Plotting the learned isotonic calibration curve

In [ ]:
r_grid_values = np.linspace(
    train_df["r_overlap"].min(),
    train_df["r_overlap"].max(),
    500
)

iso_prob_grid = isotonic_r_model.predict(r_grid_values)

plt.figure(figsize=(8, 5))

plt.plot(r_grid_values, iso_prob_grid)

plt.xlabel("r_overlap")
plt.ylabel("Predicted probability of same compound")
plt.title("Learned isotonic calibration curve")
plt.grid(alpha=0.3)
plt.show()

Comparing the learned logistic and isotonic probability curves

In [ ]:
plt.figure(figsize=(8, 5))

try:
    r_grid_df = pd.DataFrame({"r_overlap": r_grid_values})
    logistic_prob_grid = logistic_r_model.predict_proba(r_grid_df)[:, 1]
    
    plt.plot(
        r_grid_values,
        logistic_prob_grid,
        label="Logistic r_overlap"
    )
except NameError:
    pass

plt.plot(
    r_grid_values,
    iso_prob_grid,
    label="Isotonic r_overlap"
)

plt.xlabel("r_overlap")
plt.ylabel("Predicted probability of same compound")
plt.title("Learned probability curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Saving isotonic test-set predictions for later comparison

In [ ]:
isotonic_predictions_df = test_df.copy()

isotonic_predictions_df["isotonic_r_probability"] = test_iso_prob
isotonic_predictions_df["isotonic_r_prediction"] = test_iso_class

output_path = paths.processed_dir / "IdentityMF_Isotonic_r_overlap_test_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

isotonic_predictions_df.to_csv(output_path, index=False)

print("Predictions saved to:", output_path)

## Model 3: Expanded logistic calibration using within_min and maxS12

Training a logistic model using the two original min-max components as separate features

In [ ]:
expanded_features = ["within_min", "maxS12"]
target = "label"

X_train_expanded = train_df[expanded_features]
y_train_expanded = train_df[target]

X_test_expanded = test_df[expanded_features]
y_test_expanded = test_df[target]

print("Training feature shape:", X_train_expanded.shape)
print("Testing feature shape:", X_test_expanded.shape)

X_train_expanded.head()

Training the expanded logistic model without class weighting.

The model estimates:

$$
P(y=1 \mid \text{within\_min}, \text{maxS12})
$$

using:

$$
P(y=1)=\frac{1}{1+e^{-(\beta_0+\beta_1\text{within\_min}+\beta_2\text{maxS12})}}
$$

In [ ]:
expanded_logistic_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)

expanded_logistic_model.fit(X_train_expanded, y_train_expanded)

expanded_intercept = expanded_logistic_model.intercept_[0]
expanded_coefficients = expanded_logistic_model.coef_[0]

print("Intercept:", expanded_intercept)

for feature, coef in zip(expanded_features, expanded_coefficients):
    print(f"Coefficient for {feature}: {coef}")

Predicting test-set probabilities using the expanded logistic model

In [ ]:
test_expanded_prob = expanded_logistic_model.predict_proba(X_test_expanded)[:, 1]

print("First 10 predicted probabilities:")
print(test_expanded_prob[:10])

print("\nPredicted probability summary:")
print(pd.Series(test_expanded_prob).describe())

Converting expanded logistic probabilities to class labels using threshold 0.5

In [ ]:
threshold = 0.5

test_expanded_class = (test_expanded_prob >= threshold).astype(int)

print("Predicted class counts:")
print(pd.Series(test_expanded_class).value_counts().sort_index())

Evaluating expanded logistic model performance

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test_expanded, test_expanded_class).ravel()

specificity = tn / (tn + fp)
false_positive_rate = fp / (fp + tn)

expanded_metrics = {
    "accuracy": accuracy_score(y_test_expanded, test_expanded_class),
    "precision": precision_score(y_test_expanded, test_expanded_class, zero_division=0),
    "recall": recall_score(y_test_expanded, test_expanded_class),
    "specificity": specificity,
    "false_positive_rate": false_positive_rate,
    "f1_score": f1_score(y_test_expanded, test_expanded_class),
    "brier_score": brier_score_loss(y_test_expanded, test_expanded_prob),
    "log_loss": log_loss(y_test_expanded, test_expanded_prob),
    "roc_auc": roc_auc_score(y_test_expanded, test_expanded_prob),
    "pr_auc": average_precision_score(y_test_expanded, test_expanded_prob)
}

expanded_metrics_df = pd.DataFrame(
    expanded_metrics,
    index=["Expanded_logistic_within_min_maxS12"]
).T

expanded_metrics_df

Checking the expanded logistic confusion matrix at threshold 0.5

In [ ]:
expanded_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test_expanded, test_expanded_class),
    index=["Actual different (0)", "Actual same (1)"],
    columns=["Predicted different (0)", "Predicted same (1)"]
)

expanded_conf_matrix

Comparing all model metrics

In [ ]:
model_metric_tables = []

try:
    model_metric_tables.append(metrics_df)
except NameError:
    pass

try:
    model_metric_tables.append(isotonic_metrics_df)
except NameError:
    pass

model_metric_tables.append(expanded_metrics_df)

all_metrics_df = pd.concat(model_metric_tables, axis=1)

all_metrics_df

Plotting calibration curves for all available models

In [ ]:
expanded_prob_true, expanded_prob_pred = calibration_curve(
    y_test_expanded,
    test_expanded_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(6, 6))

try:
    log_prob_true, log_prob_pred = calibration_curve(
        y_test,
        test_pred_prob,
        n_bins=10,
        strategy="quantile"
    )
    plt.plot(
        log_prob_pred,
        log_prob_true,
        marker="o",
        label="Logistic r_overlap"
    )
except NameError:
    pass

try:
    iso_prob_true, iso_prob_pred = calibration_curve(
        y_test_iso,
        test_iso_prob,
        n_bins=10,
        strategy="quantile"
    )
    plt.plot(
        iso_prob_pred,
        iso_prob_true,
        marker="o",
        label="Isotonic r_overlap"
    )
except NameError:
    pass

plt.plot(
    expanded_prob_pred,
    expanded_prob_true,
    marker="o",
    label="Expanded logistic"
)

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction of same-compound rows")
plt.title("Calibration curve comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Saving expanded logistic test-set predictions

In [ ]:
expanded_predictions_df = test_df.copy()

expanded_predictions_df["expanded_logistic_probability"] = test_expanded_prob
expanded_predictions_df["expanded_logistic_prediction"] = test_expanded_class

output_path = paths.processed_dir / "IdentityMF_Expanded_logistic_test_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

expanded_predictions_df.to_csv(output_path, index=False)

print("Predictions saved to:", output_path)